In [1]:
from pathlib import Path
from itertools import islice

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from IPython.display import display

# ---------------------------------------
# Paths
# ---------------------------------------

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA = ROOT / "data"
PROCESSED = DATA / "processed" / "string_v12_0"

OUT05 = ROOT / "outputs" / "notebook_05"
OUT06 = ROOT / "outputs" / "notebook_06"

OUT06.mkdir(parents=True, exist_ok=True)

FIGURES = ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

# ---------------------------------------
# Parameters
# ---------------------------------------

THRESHOLD = 700
HLJ1_ID = "4932.YMR161W"

TARGET_GENES = [
    "SSE1",
    "APJ1",
    "SSM4",
    "YOS9",
    "USA1",
    "UBX2",
]

# ---------------------------------------
# Plot settings
# ---------------------------------------

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "figure.figsize": (10, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("Notebook 06 environment ready.")

Notebook 06 environment ready.


In [2]:
# ---------------------------------------
# 1. Load CADTP candidates
# ---------------------------------------

candidates = pd.read_csv(
    OUT05 / "final_cadtp_candidate_evidence.csv"
)

assert len(candidates) == 60
assert candidates["string_id"].is_unique

# ---------------------------------------
# 2. Load protein metadata
# ---------------------------------------

baseline = pd.read_csv(
    ROOT / "outputs" / "notebook_02"
    / "protein_baseline_features_functional_700.csv"
)

assert baseline["string_protein_id"].is_unique

# STRING ID -> gene name
name_map = (
    baseline
    .set_index("string_protein_id")["preferred_name"]
    .to_dict()
)

name_map[HLJ1_ID] = "HLJ1"

def protein_name(string_id):
    name = name_map.get(string_id)
    return str(name) if pd.notna(name) else string_id

# ---------------------------------------
# 3. Select proteins
# ---------------------------------------

targets = (
    candidates[candidates["gene"].isin(TARGET_GENES)]
    .copy()
    .sort_values(["track", "train_rank"])
)

assert set(targets["gene"]) == set(TARGET_GENES)

columns = [
    "gene",
    "orf",
    "track",
    "train_rank",
    "validation_rank",
    "node2vec_score",
    "rwr_score",
    "community_score",
    "h_hops",
    "degree",
    "physical_degree_700",
    "direct_S_neighbours",
    "jaccard_S_mean",
]

display(targets[columns])

,gene,orf,track,train_rank,validation_rank,node2vec_score,rwr_score,community_score,h_hops,degree,physical_degree_700,direct_S_neighbours,jaccard_S_mean
0,SSE1,YPL106C,A_local_consensus,1,1.0,0.998687,0.998775,0.999825,1.0,51,6,0,0.012893
1,APJ1,YNL077W,A_local_consensus,2,2.0,0.999825,0.995275,0.999125,2.0,38,5,0,0.012694
20,SSM4,YIL030C,B_cross_consensus,1,1.0,0.986699,0.995625,0.895170,1.0,34,13,5,0.380422
24,UBX2,YML013W,B_cross_consensus,5,5.0,0.977686,0.980574,0.912671,2.0,31,17,5,0.323339
28,YOS9,YDR057W,B_cross_consensus,9,9.0,0.978649,0.963423,0.929821,2.0,21,13,5,0.461286
31,USA1,YML029W,B_cross_consensus,12,12.0,0.985124,0.954323,0.915821,2.0,17,12,5,0.536519


In [4]:
# ---------------------------------------
# 1. Network loading function
# ---------------------------------------

def load_thresholded_edges(path, threshold=700):

    df = pd.read_csv(
        path,
        usecols=["u", "v", "combined_score"]
    )

    df = df.loc[
        df["combined_score"] >= threshold,
        ["u", "v", "combined_score"]
    ].copy()

    return df


# ---------------------------------------
# 2. Functional edges
# ---------------------------------------

functional_edges = load_thresholded_edges(
    PROCESSED / "functional_edges_score_ge_400.csv.gz",
    THRESHOLD
)

G_full = nx.Graph()

G_full.add_nodes_from(baseline["string_protein_id"])

G_full.add_weighted_edges_from(
    ((u, v, float(score) / 1000.0)
    for u, v, score in functional_edges.itertuples(
        index=False,
        name=None
    )),
    weight="confidence"
)

# ---------------------------------------
# 3. HLJ1 connected component
# ---------------------------------------

component = nx.node_connected_component(
    G_full,
    HLJ1_ID
)

G_F = G_full.subgraph(component).copy()

assert G_F.number_of_nodes() == 5716
assert G_F.number_of_edges() == 104138

# ---------------------------------------
# 4. Physical edges
# ---------------------------------------

physical_edges = load_thresholded_edges(
    PROCESSED / "physical_edges_score_ge_400.csv.gz",
    THRESHOLD
)

# Restrict to the same node universe
physical_edges = physical_edges[
    physical_edges["u"].isin(component)
    & physical_edges["v"].isin(component)
]

G_P = nx.Graph()
G_P.add_nodes_from(G_F.nodes())

G_P.add_weighted_edges_from(
    (
        (u, v, float(score) / 1000.0)
        for u, v, score in physical_edges.itertuples(
            index=False,
            name=None
        )
    ),
    weight="confidence"
)

# ---------------------------------------
# 5. Validate
# ---------------------------------------

assert HLJ1_ID in G_F
assert HLJ1_ID in G_P
assert set(G_F.nodes()) == set(G_P.nodes())

assert set(targets["string_id"]).issubset(G_F.nodes())

print("Network construction successful!")

print(
    f"Functional: {G_F.number_of_nodes():,} nodes, "
    f"{G_F.number_of_edges():,} edges"
)

print(
    f"Physical: {G_P.number_of_nodes():,} nodes, "
    f"{G_P.number_of_edges():,} edges"
)

print(f"HLJ1 functional degree: {G_F.degree(HLJ1_ID)}")
print(f"HLJ1 physical degree: {G_P.degree(HLJ1_ID)}")

Network construction successful!
Functional: 5,716 nodes, 104,138 edges
Physical: 5,716 nodes, 43,028 edges
HLJ1 functional degree: 25
HLJ1 physical degree: 0


In [5]:
# ---------------------------------------
# 1. Helper functions
# ---------------------------------------

def path_bottleneck(G, path):
    """Minimum edge confidence along a path."""

    return min(
        G[u][v]["confidence"]
        for u, v in zip(path[:-1], path[1:])
    )


def explain_shortest_paths(G, source, target):

    if not nx.has_path(G, source, target):
        return {
            "distance": np.nan,
            "num_shortest_paths": 0,
            "best_path": None,
            "best_bottleneck": np.nan,
        }

    # Enumerate all hop-shortest paths
    paths = list(
        nx.all_shortest_paths(
            G,
            source=source,
            target=target
        )
    )

    # Select path with highest bottleneck
    best_path = max(
        paths,
        key=lambda p: (
            path_bottleneck(G, p),
            tuple(p)
        )
    )

    return {
        "distance": len(best_path) - 1,
        "num_shortest_paths": len(paths),
        "best_path": " → ".join(
            protein_name(v) for v in best_path
        ),
        "best_bottleneck": path_bottleneck(
            G, best_path
        ),
    }


# ---------------------------------------
# 2. Analyse selected proteins
# ---------------------------------------

path_records = []

for row in targets.itertuples(index=False):

    result = explain_shortest_paths(
        G_F,
        HLJ1_ID,
        row.string_id
    )

    path_records.append({
        "gene": row.gene,
        "orf": row.orf,
        "track": row.track,
        **result
    })

path_results = pd.DataFrame(path_records)

display(path_results.sort_values(["distance", "gene"]))

# ---------------------------------------
# 3. Export
# ---------------------------------------

path_results.to_csv(
    OUT06 / "candidate_shortest_path_evidence.csv",
    index=False
)

,gene,orf,track,distance,num_shortest_paths,best_path,best_bottleneck
0,SSE1,YPL106C,A_local_consensus,1,1,HLJ1 → SSE1,0.831
2,SSM4,YIL030C,B_cross_consensus,1,1,HLJ1 → SSM4,0.735
1,APJ1,YNL077W,A_local_consensus,2,22,HLJ1 → SWA2 → APJ1,0.941
3,UBX2,YML013W,B_cross_consensus,2,3,HLJ1 → UFD2 → UBX2,0.741
5,USA1,YML029W,B_cross_consensus,2,2,HLJ1 → KAR2 → USA1,0.894
4,YOS9,YDR057W,B_cross_consensus,2,2,HLJ1 → KAR2 → YOS9,0.894


In [6]:
# ---------------------------------------
# 1. Direct HLJ1 associations
# ---------------------------------------

records = []

for row in targets.itertuples(index=False):

    v = row.string_id

    functional_direct = G_F.has_edge(HLJ1_ID, v)
    physical_direct = G_P.has_edge(HLJ1_ID, v)

    physical_connected = nx.has_path(
        G_P, HLJ1_ID, v
    )

    physical_distance = (
        nx.shortest_path_length(
            G_P, HLJ1_ID, v
        )
        if physical_connected
        else np.nan
    )

    records.append({
        "gene": row.gene,
        "functional_distance": nx.shortest_path_length(
            G_F, HLJ1_ID, v
        ),
        "functional_direct": functional_direct,
        "physical_direct": physical_direct,
        "physical_distance": physical_distance,
        "functional_degree": G_F.degree(v),
        "physical_degree": G_P.degree(v),
        "functional_edge_score": (
            G_F[HLJ1_ID][v]["confidence"]
            if functional_direct else np.nan
        ),
        "physical_edge_score": (
            G_P[HLJ1_ID][v]["confidence"]
            if physical_direct else np.nan
        ),
    })

cross_layer_evidence = pd.DataFrame(records)

display(cross_layer_evidence)

cross_layer_evidence.to_csv(
    OUT06 / "candidate_cross_layer_evidence.csv",
    index=False
)

,gene,functional_distance,functional_direct,physical_direct,physical_distance,functional_degree,physical_degree,functional_edge_score,physical_edge_score
0,SSE1,1,True,False,NaN,51,6,0.831,NaN
1,APJ1,2,False,False,NaN,38,5,NaN,NaN
2,SSM4,1,True,False,NaN,34,13,0.735,NaN
3,UBX2,2,False,False,NaN,31,17,NaN,NaN
4,YOS9,2,False,False,NaN,21,13,NaN,NaN
5,USA1,2,False,False,NaN,17,12,NaN,NaN


In [7]:
# ---------------------------------------
# 1. Predefined ERAD reference proteins
# ---------------------------------------

ERAD_REFERENCES = {
    "HRD1": "YOL013C",
    "DER1": "YBR201W",
    "HRD3": "YLR207W",
    "UBC7": "YMR022W",
    "CUE1": "YMR264W",
}

# ---------------------------------------
# 2. Check direct associations
# ---------------------------------------

records = []

for row in targets.itertuples(index=False):

    v = row.string_id

    for gene, orf in ERAD_REFERENCES.items():

        ref_id = f"4932.{orf}"

        functional_direct = G_F.has_edge(v, ref_id)

        physical_direct = G_P.has_edge(v, ref_id)

        records.append({
            "candidate": row.gene,
            "reference": gene,
            "functional_direct": functional_direct,
            "physical_direct": physical_direct,
            "functional_confidence": (
                G_F[v][ref_id]["confidence"]
                if functional_direct else np.nan
            ),
            "physical_confidence": (
                G_P[v][ref_id]["confidence"]
                if physical_direct else np.nan
            ),
        })

erad_associations = pd.DataFrame(records)

# ---------------------------------------
# 3. Summary
# ---------------------------------------

erad_summary = (
    erad_associations
    .groupby("candidate")
    .agg(
        functional_seed_connections=(
            "functional_direct", "sum"
        ),
        physical_seed_connections=(
            "physical_direct", "sum"
        ),
        mean_functional_confidence=(
            "functional_confidence", "mean"
        ),
    )
    .reset_index()
)

display(erad_summary)

# ---------------------------------------
# 4. Detailed evidence
# ---------------------------------------

display(erad_associations.sort_values(
        ["candidate", "reference"]
    )
)

erad_associations.to_csv(
    OUT06 / "candidate_erad_associations.csv",
    index=False
)

,candidate,functional_seed_connections,physical_seed_connections,mean_functional_confidence
0,APJ1,0,0,NaN
1,SSE1,0,0,NaN
2,SSM4,5,5,0.9966
3,UBX2,5,5,0.9972
4,USA1,5,5,0.9920
5,YOS9,5,5,0.9938


,candidate,reference,functional_direct,physical_direct,functional_confidence,physical_confidence
9,APJ1,CUE1,False,False,NaN,NaN
6,APJ1,DER1,False,False,NaN,NaN
5,APJ1,HRD1,False,False,NaN,NaN
7,APJ1,HRD3,False,False,NaN,NaN
8,APJ1,UBC7,False,False,NaN,NaN
4,SSE1,CUE1,False,False,NaN,NaN
1,SSE1,DER1,False,False,NaN,NaN
0,SSE1,HRD1,False,False,NaN,NaN
2,SSE1,HRD3,False,False,NaN,NaN
3,SSE1,UBC7,False,False,NaN,NaN


In [8]:
# --------------------------------------------------
# 1. Load original functional edge list
# --------------------------------------------------

all_functional_edges = pd.read_csv(
    PROCESSED / "functional_edges_score_ge_400.csv.gz",
    usecols=["u", "v", "combined_score"]
)

THRESHOLDS = [400, 600, 700, 750, 800, 900]

records = []

# --------------------------------------------------
# 2. Evaluate each threshold
# --------------------------------------------------

for tau in THRESHOLDS:

    print(f"Evaluating threshold {tau}...")

    selected = all_functional_edges.loc[
        all_functional_edges["combined_score"] >= tau
    ]

    G_tau = nx.Graph()

    # Keep the complete original node universe
    G_tau.add_nodes_from(
        baseline["string_protein_id"]
    )

    G_tau.add_edges_from(
        selected[["u", "v"]].itertuples(
            index=False,
            name=None
        )
    )

    distances_tau = (
        nx.single_source_shortest_path_length(
            G_tau, HLJ1_ID
        )
    )

    for row in targets.itertuples(index=False):

        v = row.string_id

        records.append({
            "threshold": tau,
            "gene": row.gene,
            "degree": G_tau.degree(v),
            "distance": distances_tau.get(
                v, np.nan
            ),
            "direct_hlj1_edge": G_tau.has_edge(
                HLJ1_ID, v
            ),
            "connected_to_hlj1":
                v in distances_tau,
        })

# --------------------------------------------------
# 3. Results
# --------------------------------------------------

threshold_results = pd.DataFrame(records)

display(
    threshold_results.pivot(
        index="gene",
        columns="threshold",
        values="distance"
    )
)

print("\nDirect associations with HLJ1")

display(
    threshold_results.pivot(
        index="gene",
        columns="threshold",
        values="direct_hlj1_edge"
    )
)

threshold_results.to_csv(
    OUT06 / "candidate_threshold_sensitivity.csv",
    index=False
)

Evaluating threshold 400...
Evaluating threshold 600...
Evaluating threshold 700...
Evaluating threshold 750...
Evaluating threshold 800...
Evaluating threshold 900...


threshold,400,600,700,750,800,900
gene,,,,,,
APJ1,1,2,2,2,2,2
SSE1,1,1,1,1,1,3
SSM4,1,1,1,3,3,4
UBX2,2,2,2,3,3,5
USA1,2,2,2,2,2,4
YOS9,2,2,2,2,2,4



Direct associations with HLJ1


threshold,400,600,700,750,800,900
gene,,,,,,
APJ1,True,False,False,False,False,False
SSE1,True,True,True,True,True,False
SSM4,True,True,True,False,False,False
UBX2,False,False,False,False,False,False
USA1,False,False,False,False,False,False
YOS9,False,False,False,False,False,False


In [9]:
# ---------------------------------------
# 1. Curated biological annotations
# ---------------------------------------

annotations = pd.DataFrame([
    {
        "gene": "SSE1",
        "biological_role": "Hsp110-family chaperone",
        "biological_theme": "Protein folding",
        "source_url": "https://www.yeastgenome.org/locus/YPL106C",
    },
    {
        "gene": "APJ1",
        "biological_role": "Hsp40/DnaJ-family chaperone",
        "biological_theme": "Protein quality control",
        "source_url": "https://www.yeastgenome.org/locus/YNL077W",
    },
    {
        "gene": "SSM4",
        "biological_role": "ER-associated E3 ubiquitin ligase",
        "biological_theme": "ERAD",
        "source_url": "https://www.ncbi.nlm.nih.gov/gene/854781",
    },
    {
        "gene": "UBX2",
        "biological_role": "ERAD-associated adaptor protein",
        "biological_theme": "ERAD",
        "source_url": "https://www.ncbi.nlm.nih.gov/gene/854995",
    },
    {
        "gene": "YOS9",
        "biological_role": "ER quality-control lectin",
        "biological_theme": "ERAD substrate recognition",
        "source_url": "https://wiki.yeastgenome.org/index.php/YDR057W",
    },
    {
        "gene": "USA1",
        "biological_role": "Hrd1 ubiquitin ligase complex scaffold",
        "biological_theme": "ERAD",
        "source_url": "https://yeastract-plus.org/yeastract/scerevisiae/view.php?existing=locus&orfname=YML029W",
    },
]).set_index("gene")

# ---------------------------------------
# 2. Combine computational evidence
# ---------------------------------------

profile = (
    targets
    .set_index("gene")[
        [
            "orf",
            "track",
            "train_rank",
            "validation_rank",
            "degree",
            "physical_degree_700",
        ]
    ]
    .join(
        path_results.set_index("gene")[
            ["num_shortest_paths", "best_path"]
        ]
    )
    .join(
        erad_summary.set_index("candidate")[
            [
                "functional_seed_connections",
                "physical_seed_connections",
            ]
        ]
    )
    .join(annotations)
)

# ---------------------------------------
# 3. Add threshold sensitivity
# ---------------------------------------

distance_table = threshold_results.pivot(
    index="gene",
    columns="threshold",
    values="distance"
)

distance_table.columns = [
    f"distance_{tau}"
    for tau in distance_table.columns
]

profile = profile.join(distance_table)

assert len(profile) == 6
assert profile["biological_role"].notna().all()

# ---------------------------------------
# 4. Export
# ---------------------------------------

profile.to_csv(
    OUT06 / "candidate_explanation_profiles.csv"
)

display(profile)

,orf,track,train_rank,validation_rank,degree,physical_degree_700,num_shortest_paths,best_path,functional_seed_connections,physical_seed_connections,biological_role,biological_theme,source_url,distance_400,distance_600,distance_700,distance_750,distance_800,distance_900
gene,,,,,,,,,,,,,,,,,,,
SSE1,YPL106C,A_local_consensus,1,1.0,51,6,1,HLJ1 → SSE1,0,0,Hsp110-family chaperone,Protein folding,https://www.yeastgenome.org/locus/YPL106C,1,1,1,1,1,3
APJ1,YNL077W,A_local_consensus,2,2.0,38,5,22,HLJ1 → SWA2 → APJ1,0,0,Hsp40/DnaJ-family chaperone,Protein quality control,https://www.yeastgenome.org/locus/YNL077W,1,2,2,2,2,2
SSM4,YIL030C,B_cross_consensus,1,1.0,34,13,1,HLJ1 → SSM4,5,5,ER-associated E3 ubiquitin ligase,ERAD,https://www.ncbi.nlm.nih.gov/gene/854781,1,1,1,3,3,4
UBX2,YML013W,B_cross_consensus,5,5.0,31,17,3,HLJ1 → UFD2 → UBX2,5,5,ERAD-associated adaptor protein,ERAD,https://www.ncbi.nlm.nih.gov/gene/854995,2,2,2,3,3,5
YOS9,YDR057W,B_cross_consensus,9,9.0,21,13,2,HLJ1 → KAR2 → YOS9,5,5,ER quality-control lectin,ERAD substrate recognition,https://wiki.yeastgenome.org/index.php/YDR057W,2,2,2,2,2,4
USA1,YML029W,B_cross_consensus,12,12.0,17,12,2,HLJ1 → KAR2 → USA1,5,5,Hrd1 ubiquitin ligase complex scaffold,ERAD,https://yeastract-plus.org/yeastract/scerevisi...,2,2,2,2,2,4


In [11]:
from pathlib import Path
from urllib.request import urlretrieve
from datetime import datetime, timezone
import hashlib
import gzip
import re

import pandas as pd
import numpy as np

# ---------------------------------------
# 1. Directories
# ---------------------------------------

GO_DIR = DATA / "raw" / "go"
GO_DIR.mkdir(parents=True, exist_ok=True)

GAF_PATH = GO_DIR / "YEAST-mod.gaf"
OBO_PATH = GO_DIR / "go-basic.obo"

# ---------------------------------------
# 2. Official sources
# ---------------------------------------

GO_URLS = {
    GAF_PATH: (
        "https://current.geneontology.org/"
        "annotations/gaf/YEAST-mod.gaf.gz"
    ),
    OBO_PATH: (
        "https://current.geneontology.org/"
        "ontology/go-basic.obo"
    ),
}

for path, url in GO_URLS.items():

    if not path.exists():
        print(f"Downloading {path.name}...")
        urlretrieve(url, path)

    print(
        f"{path.name}: "
        f"{path.stat().st_size / 1024**2:.2f} MB"
    )

# ---------------------------------------
# 3. Record provenance
# ---------------------------------------

for path in [GAF_PATH, OBO_PATH]:

    digest = hashlib.file_digest(
        open(path, "rb"), "sha256"
    ).hexdigest()

    print(f"{path.name} SHA256: {digest}")

print(
    "Download checked at:",
    datetime.now(timezone.utc).isoformat()
)

YEAST-mod.gaf: 24.63 MB
go-basic.obo: 30.73 MB
YEAST-mod.gaf SHA256: 504e436d4e7baa370d819e1d77d4c9a251802cb0008c1dcdbf2f376eb5c6bc52
go-basic.obo SHA256: b08d45b268b8c24ccb2513dbbbc7d4df9f6521c099b413f79eb31e06e0fa3bcc
Download checked at: 2026-10-19T06:04:50.845332+00:00


In [12]:
# ---------------------------------------
# 1. Load GAF
# ---------------------------------------

gaf = pd.read_csv(
    GAF_PATH,
    sep="\t",
    comment="!",
    header=None,
    dtype=str,
    usecols=[1, 2, 3, 4, 6, 8, 10],
    low_memory=False
)

gaf.columns = [
    "sgd_id",
    "symbol",
    "qualifier",
    "go_id",
    "evidence",
    "aspect",
    "synonyms",
]

print("Raw annotation records:", len(gaf))

# ---------------------------------------
# 2. Exclude negative annotations
# ---------------------------------------

is_negative = gaf["qualifier"].fillna("").apply(
    lambda x: "NOT" in x.split("|")
)

gaf = gaf.loc[
    ~is_negative
    & (gaf["aspect"] == "P")
    & (~gaf["evidence"].isin(["IEA", "ND"]))
].copy()

# ---------------------------------------
# 3. Map GAF records to systematic ORFs
# ---------------------------------------

# Nuclear systematic ORFs:
# YPL106C, YNL077W, YJR135W-A, etc.

ORF_PATTERN = r"(Y[A-P][LR]\d{3}[CW](?:-[A-Z])?)"

identifiers = gaf["symbol"].fillna("") + "|" + gaf["synonyms"].fillna("")

gaf["orf"] = identifiers.str.extract(
    ORF_PATTERN,
    expand=False
)

# Exclude records without a mapped systematic ORF
gaf = gaf.dropna(subset=["orf"]).copy()

print("Filtered BP annotation records:", len(gaf))
print("Unique annotated ORFs:", gaf["orf"].nunique())

# ---------------------------------------
# 4. Inspect our six candidates
# ---------------------------------------

target_orfs = targets["orf"].tolist()

target_go = gaf.loc[gaf["orf"].isin(target_orfs)].copy()

print("\nGO records for selected proteins:")

display(
    target_go.groupby("orf")
    .agg(
        num_annotations=("go_id", "size"),
        unique_terms=("go_id", "nunique"),
        evidence_types=("evidence", lambda s: ", ".join(sorted(set(s))))
    )
)

Raw annotation records: 142056
Filtered BP annotation records: 31590
Unique annotated ORFs: 5124

GO records for selected proteins:


,num_annotations,unique_terms,evidence_types
orf,,,
YDR057W,13,6,"IBA, IMP, NAS"
YIL030C,7,3,"IBA, IDA, IMP, NAS"
YML013W,17,5,"IBA, IGI, IMP, IPI, NAS"
YML029W,6,3,"IDA, IGI, IMP, IPI, NAS"
YNL077W,4,4,"IBA, IGI, IMP"
YPL106C,6,5,"IBA, IDA, IGI, IMP"


In [13]:
%pip install goatools

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 13.4 MB/s  0:00:01 eta 0:00:01
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
Using cached rich-15.0.0-py3-none-any.whl (310 kB)
Using cached markdown_it_py-4.2.0-py3-none-any.whl (91 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 20.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 15.3 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [goatools]/15 [goatools]ls]
Note: you may need to restart the kernel to use updated packages

In [14]:
from goatools.obo_parser import GODag

# ---------------------------------------
# 1. Load GO ontology
# ---------------------------------------

go = GODag(str(OBO_PATH))

# ---------------------------------------
# 2. Define categories
# ---------------------------------------

GO_CATEGORIES = {
    "protein_folding": "GO:0006457",
    "ERAD": "GO:0036503",
    "ER_stress": "GO:0034976",
    "unfolded_protein_response": "GO:0006986",
}

# ---------------------------------------
# 3. Expand to descendants
# ---------------------------------------

go_term_sets = {}

for category, go_id in GO_CATEGORIES.items():

    assert go_id in go, f"Missing GO term: {go_id}"

    descendants = go[go_id].get_all_children()

    go_term_sets[category] = {go_id} | descendants

    print(f"{category}: {len(go_term_sets[category])} terms")

# ---------------------------------------
# 4. GO terms per protein
# ---------------------------------------

go_by_orf = (
    gaf.groupby("orf")["go_id"]
    .agg(set)
    .to_dict()
)

# ---------------------------------------
# 5. Construct binary biological labels
# ---------------------------------------

all_orfs = (
    baseline["orf"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

go_labels = pd.DataFrame(index=all_orfs)
go_labels.index.name = "orf"

go_labels["has_BP_annotation"] = [
    orf in go_by_orf
    for orf in go_labels.index
]

for category, terms in go_term_sets.items():

    go_labels[category] = [
        bool(go_by_orf.get(orf, set()) & terms)
        for orf in go_labels.index
    ]

# Composite outcome: any predefined relevant category
go_labels["proteostasis_any"] = (
    go_labels[list(GO_CATEGORIES)]
    .any(axis=1)
)

# ---------------------------------------
# 6. Inspect candidate labels
# ---------------------------------------

target_labels = (
    targets[["gene", "orf", "track", "train_rank"]]
    .merge(
        go_labels.reset_index(),
        on="orf",
        how="left",
        validate="many_to_one"
    )
)

display(target_labels.sort_values(
    ["track", "train_rank"]
))

# ---------------------------------------
# 7. Annotation coverage
# ---------------------------------------

print("GO annotation coverage")

print(go_labels.mean(numeric_only=True).round(4))

# ---------------------------------------
# 8. Save
# ---------------------------------------

go_labels.to_csv(OUT06 / "go_biological_process_labels.csv")

/Users/kennyyu/math3888-stream-2-group-k/data/raw/go/go-basic.obo: fmt(1.2) rel(2026-07-26) 41,378 Terms
protein_folding: 6 terms
ERAD: 2 terms
ER_stress: 14 terms
unfolded_protein_response: 4 terms


,gene,orf,track,train_rank,has_BP_annotation,protein_folding,ERAD,ER_stress,unfolded_protein_response,proteostasis_any
0,SSE1,YPL106C,A_local_consensus,1,True,True,False,False,False,True
1,APJ1,YNL077W,A_local_consensus,2,True,True,False,False,False,True
2,SSM4,YIL030C,B_cross_consensus,1,True,False,True,True,False,True
3,UBX2,YML013W,B_cross_consensus,5,True,False,True,True,False,True
4,YOS9,YDR057W,B_cross_consensus,9,True,False,True,True,True,True
5,USA1,YML029W,B_cross_consensus,12,True,False,True,True,False,True


GO annotation coverage
has_BP_annotation            0.7756
protein_folding              0.0189
ERAD                         0.0088
ER_stress                    0.0132
unfolded_protein_response    0.0024
proteostasis_any             0.0315
dtype: float64


In [15]:
# ---------------------------------------
# 1. Load full CADTP scores
# ---------------------------------------

scores = pd.read_csv(
    OUT05 / "cadtp_full_candidate_scores.csv.gz",
    index_col="string_id"
)

# ---------------------------------------
# 2. Load GO biological labels
# ---------------------------------------

labels = pd.read_csv(
    OUT06 / "go_biological_process_labels.csv"
).set_index("orf")

# Join using systematic ORF
evaluation = scores.join(
    labels,
    on="orf",
    validate="many_to_one"
)

# ---------------------------------------
# 3. Data validation
# ---------------------------------------

assert len(evaluation) == 4439
assert evaluation["orf"].is_unique

assert evaluation[
    ["N", "R", "hybrid_score", "degree", "h_hops"]
].notna().all().all()

GO_COLUMNS = [
    "has_BP_annotation",
    "protein_folding",
    "ERAD",
    "ER_stress",
    "unfolded_protein_response",
    "proteostasis_any",
]

# Unmapped ORFs are recorded before filling labels
evaluation["go_mapping_available"] = (
    evaluation["proteostasis_any"].notna()
)

evaluation[GO_COLUMNS] = (
    evaluation[GO_COLUMNS]
    .fillna(False)
    .astype(bool)
)

# ---------------------------------------
# 4. Summary
# ---------------------------------------

print("Eligible proteins:", len(evaluation))

display(
    evaluation.groupby("consensus_track").agg(
        candidates=("orf", "size"),
        BP_coverage=("has_BP_annotation", "mean"),
        proteostasis_rate=("proteostasis_any", "mean"),
        ERAD_rate=("ERAD", "mean"),
    )
)

print(
    "GO mapping coverage:",
    f"{evaluation['go_mapping_available'].mean():.2%}"
)

Eligible proteins: 4439


,candidates,BP_coverage,proteostasis_rate,ERAD_rate
consensus_track,,,,
A_local_consensus,120,0.866667,0.491667,0.058333
B_cross_consensus,3985,0.831117,0.020828,0.007528
Boundary,334,0.823353,0.053892,0.008982


GO mapping coverage: 100.00%


In [16]:
# ---------------------------------------
# 1. Settings
# ---------------------------------------

METHODS = {
    "Degree": "degree",
    "Node2Vec": "N",
    "RWR": "R",
    "CADTP": "hybrid_score",
}

TRACKS = [
    "A_local_consensus",
    "B_cross_consensus",
]

K_VALUES = [10, 20, 50]

# ---------------------------------------
# 2. Baseline evaluation
# ---------------------------------------

records = []

for track in TRACKS:

    pool = evaluation.loc[
        evaluation["consensus_track"] == track
    ].copy()

    background_rate = pool["proteostasis_any"].mean()

    for method, score_col in METHODS.items():

        ranked = pool.sort_values(
            [score_col, "orf"],
            ascending=[False, True],
            kind="mergesort"
        )

        for k in K_VALUES:

            if len(ranked) < k:
                continue

            top = ranked.head(k)

            hits = int(top["proteostasis_any"].sum())

            precision = hits / k

            fold = (
                precision / background_rate
                if background_rate > 0
                else np.nan
            )

            records.append({
                "track": track,
                "method": method,
                "k": k,
                "hits": hits,
                "precision_at_k": precision,
                "background_rate": background_rate,
                "fold_enrichment": fold,
                "topk_BP_coverage":
                    top["has_BP_annotation"].mean(),
            })

baseline_results = pd.DataFrame(records)

# ---------------------------------------
# 3. Display
# ---------------------------------------

for track in TRACKS:

    print(f"\n{track}")

    display(
        baseline_results.loc[
            (baseline_results["track"] == track)
            & (baseline_results["k"] == 20)
        ]
        .sort_values(
            "precision_at_k",
            ascending=False
        )
    )

# ---------------------------------------
# 4. Save
# ---------------------------------------

baseline_results.to_csv(
    OUT06 / "go_baseline_comparison.csv",
    index=False
)


A_local_consensus


,track,method,k,hits,precision_at_k,background_rate,fold_enrichment,topk_BP_coverage
1,A_local_consensus,Degree,20,18,0.90,0.491667,1.830508,1.00
7,A_local_consensus,RWR,20,16,0.80,0.491667,1.627119,0.95
10,A_local_consensus,CADTP,20,16,0.80,0.491667,1.627119,0.95
4,A_local_consensus,Node2Vec,20,15,0.75,0.491667,1.525424,0.95



B_cross_consensus


,track,method,k,hits,precision_at_k,background_rate,fold_enrichment,topk_BP_coverage
16,B_cross_consensus,Node2Vec,20,11,0.55,0.020828,26.406627,0.9
22,B_cross_consensus,CADTP,20,10,0.50,0.020828,24.006024,1.0
19,B_cross_consensus,RWR,20,4,0.20,0.020828,9.602410,1.0
13,B_cross_consensus,Degree,20,0,0.00,0.020828,0.000000,1.0


In [17]:
# ---------------------------------------
# 1. Settings
# ---------------------------------------

N_PERMUTATIONS = 2000
RANDOM_SEED = 3888

rng = np.random.default_rng(RANDOM_SEED)

# ---------------------------------------
# 2. Degree and distance strata
# ---------------------------------------

evaluation["degree_bin"] = pd.cut(
    evaluation["degree"],
    bins=[-np.inf, 5, 20, 60, np.inf],
    labels=["0-5", "6-20", "21-60", "61+"]
)

evaluation["distance_bin"] = (
    evaluation["h_hops"]
    .clip(upper=4)
    .astype(int)
    .astype(str)
)

evaluation["stratum"] = (
    evaluation["distance_bin"]
    + "_"
    + evaluation["degree_bin"].astype(str)
)

# ---------------------------------------
# 3. Matched randomisation function
# ---------------------------------------

def matched_random_test(
    pool,
    selected,
    label="proteostasis_any",
    n_permutations=2000,
):

    # Number of selected proteins in each stratum
    quotas = selected["stratum"].value_counts()

    # Candidate IDs available per stratum
    groups = {
        s: g.index.to_numpy()
        for s, g in pool.groupby("stratum")
    }

    observed = int(selected[label].sum())

    simulated = np.empty(
        n_permutations,
        dtype=int
    )

    # Pre-compute GO labels
    label_map = pool[label].to_dict()

    for b in range(n_permutations):

        sampled = []

        for stratum, count in quotas.items():

            choices = rng.choice(
                groups[stratum],
                size=int(count),
                replace=False
            )

            sampled.extend(choices)

        simulated[b] = sum(
            label_map[v]
            for v in sampled
        )

    empirical_p = (
        1 + np.count_nonzero(simulated >= observed)
    ) / (n_permutations + 1)

    return {
        "observed_hits": observed,
        "null_mean_hits": simulated.mean(),
        "null_std_hits": simulated.std(ddof=1),
        "matched_fold_enrichment": (
            observed / simulated.mean()
            if simulated.mean() > 0
            else np.nan
        ),
        "empirical_p_value": empirical_p,
    }


# ---------------------------------------
# 4. Run matched tests
# ---------------------------------------

matched_records = []

for track in TRACKS:

    pool = evaluation.loc[
        evaluation["consensus_track"] == track
    ].copy()

    for method, score_col in METHODS.items():

        selected = (
            pool.sort_values(
                [score_col, "orf"],
                ascending=[False, True],
                kind="mergesort"
            )
            .head(20)
        )

        result = matched_random_test(
            pool=pool,
            selected=selected,
            n_permutations=N_PERMUTATIONS
        )

        matched_records.append({
            "track": track,
            "method": method,
            "k": 20,
            **result
        })

matched_results = pd.DataFrame(matched_records)

# ---------------------------------------
# 5. Display results
# ---------------------------------------

for track in TRACKS:

    print(f"\n{track}")

    display(
        matched_results.loc[
            matched_results["track"] == track
        ].sort_values(
            "matched_fold_enrichment",
            ascending=False
        )
    )

matched_results.to_csv(
    OUT06 / "go_matched_randomisation.csv",
    index=False
)


A_local_consensus


,track,method,k,observed_hits,null_mean_hits,null_std_hits,matched_fold_enrichment,empirical_p_value
0,A_local_consensus,Degree,20,18,16.0120,1.325041,1.124157,0.131434
1,A_local_consensus,Node2Vec,20,15,14.0355,1.537017,1.068719,0.370315
3,A_local_consensus,CADTP,20,16,15.2515,1.266511,1.049077,0.423288
2,A_local_consensus,RWR,20,16,15.5700,0.570318,1.027617,0.610195



B_cross_consensus


,track,method,k,observed_hits,null_mean_hits,null_std_hits,matched_fold_enrichment,empirical_p_value
5,B_cross_consensus,Node2Vec,20,11,5.9915,1.397284,1.835934,0.001499
7,B_cross_consensus,CADTP,20,10,6.2190,1.500722,1.607976,0.015492
6,B_cross_consensus,RWR,20,4,4.0980,1.221942,0.976086,0.670665
4,B_cross_consensus,Degree,20,0,0.8595,0.832530,0.000000,1.000000


In [18]:
from statsmodels.stats.multitest import multipletests

# ---------------------------------------
# 1. Adjust for multiple testing
# ---------------------------------------

statistical_results = matched_results.copy()

statistical_results["q_value_BH"] = multipletests(
    statistical_results["empirical_p_value"],
    alpha=0.05,
    method="fdr_bh"
)[1]

statistical_results["significant_FDR_05"] = (
    statistical_results["q_value_BH"] < 0.05
)

# ---------------------------------------
# 2. Display
# ---------------------------------------

display(
    statistical_results.sort_values(
        "empirical_p_value"
    )
)

# ---------------------------------------
# 3. Export
# ---------------------------------------

statistical_results.to_csv(
    OUT06 / "go_matched_tests_adjusted.csv",
    index=False
)

,track,method,k,observed_hits,null_mean_hits,null_std_hits,matched_fold_enrichment,empirical_p_value,q_value_BH,significant_FDR_05
5,B_cross_consensus,Node2Vec,20,11,5.9915,1.397284,1.835934,0.001499,0.011994,True
7,B_cross_consensus,CADTP,20,10,6.2190,1.500722,1.607976,0.015492,0.061969,False
0,A_local_consensus,Degree,20,18,16.0120,1.325041,1.124157,0.131434,0.350491,False
1,A_local_consensus,Node2Vec,20,15,14.0355,1.537017,1.068719,0.370315,0.677261,False
3,A_local_consensus,CADTP,20,16,15.2515,1.266511,1.049077,0.423288,0.677261,False
2,A_local_consensus,RWR,20,16,15.5700,0.570318,1.027617,0.610195,0.766474,False
6,B_cross_consensus,RWR,20,4,4.0980,1.221942,0.976086,0.670665,0.766474,False
4,B_cross_consensus,Degree,20,0,0.8595,0.832530,0.000000,1.000000,1.000000,False


In [19]:
# ---------------------------------------
# 1. Evaluate individual GO categories
# ---------------------------------------

CATEGORIES = [
    "protein_folding",
    "ERAD",
    "ER_stress",
    "unfolded_protein_response",
]

records = []

for track in TRACKS:

    pool = evaluation.loc[
        evaluation["consensus_track"] == track
    ].copy()

    for method, score_col in METHODS.items():

        ranked = pool.sort_values(
            [score_col, "orf"],
            ascending=[False, True],
            kind="mergesort"
        )

        for k in [10, 20, 50]:

            top = ranked.head(k)

            for category in CATEGORIES:

                records.append({
                    "track": track,
                    "method": method,
                    "k": k,
                    "category": category,
                    "hits": int(top[category].sum()),
                    "precision": top[category].mean(),
                    "background_rate": pool[category].mean(),
                })

category_results = pd.DataFrame(records)

# ---------------------------------------
# 2. Focus on CADTP Top-20
# ---------------------------------------

cadtp_categories = category_results.loc[
    (category_results["method"] == "CADTP")
    & (category_results["k"] == 20)
]

display(
    cadtp_categories.pivot(
        index="category",
        columns="track",
        values="hits"
    )
)

# ---------------------------------------
# 3. Compare cross-community methods
# ---------------------------------------

cross_categories = category_results.loc[
    (category_results["track"] == "B_cross_consensus")
    & (category_results["k"] == 20)
]

display(
    cross_categories.pivot(
        index="category",
        columns="method",
        values="hits"
    )
)

# ---------------------------------------
# 4. Export
# ---------------------------------------

category_results.to_csv(
    OUT06 / "go_category_comparison.csv",
    index=False
)

track,A_local_consensus,B_cross_consensus
category,,
ERAD,2,9
ER_stress,2,9
protein_folding,15,1
unfolded_protein_response,2,2


method,CADTP,Degree,Node2Vec,RWR
category,,,,
ERAD,9,0,11,4
ER_stress,9,0,11,4
protein_folding,1,0,0,0
unfolded_protein_response,2,0,2,0
